In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve, auc, precision_recall_curve,
                             average_precision_score)

import shap

plt.style.use('fivethirtyeight')
sns.set_palette("Set2")

In [2]:
# Load data
df = pd.read_csv('fraud_oracle.csv')

print("Data Shape:", df.shape)
print("\nFirst few rows of data:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nTarget Variable Distribution:")
if 'fraud_reported' in df.columns:
    fraud_counts = df['fraud_reported'].value_counts()
    print(fraud_counts)
    print(f"Fraud Rate: {fraud_counts['Y'] / len(df) * 100:.2f}%")
else:
    print("Target variable 'fraud_reported' not found. Checking for similar columns:")
    print(df.columns.tolist())

missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percent})
missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values('Percentage', ascending=False)

print("\nColumns with missing values:")
print(missing_df)

Data Shape: (15420, 33)

First few rows of data:
  Month  WeekOfMonth  DayOfWeek    Make AccidentArea DayOfWeekClaimed  \
0   Dec            5  Wednesday   Honda        Urban          Tuesday   
1   Jan            3  Wednesday   Honda        Urban           Monday   
2   Oct            5     Friday   Honda        Urban         Thursday   
3   Jun            2   Saturday  Toyota        Rural           Friday   
4   Jan            5     Monday   Honda        Urban          Tuesday   

  MonthClaimed  WeekOfMonthClaimed     Sex MaritalStatus  ...  AgeOfVehicle  \
0          Jan                   1  Female        Single  ...       3 years   
1          Jan                   4    Male        Single  ...       6 years   
2          Nov                   2    Male       Married  ...       7 years   
3          Jul                   1    Male       Married  ...   more than 7   
4          Feb                   2  Female        Single  ...       5 years   

  AgeOfPolicyHolder PoliceReportFiled

In [5]:
print("Test phase___")

data = df.copy()

target_col = 'fraud_reported' if 'fraud_reported' in data.columns else data.columns[-1]
print(f"Using '{target_col}' as target variable")

if data[target_col].dtype == 'object':
    data[target_col] = data[target_col].map({'Y': 1, 'N': 0, 'Yes': 1, 'No': 0})
else:
    data[target_col] = data[target_col].astype(int)

categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
if target_col in categorical_cols:
    categorical_cols.remove(target_col)

numerical_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numerical columns: {len(numerical_cols)}")

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

print("Categorical variables encoded.")

if 'total_claim_amount' in data.columns and 'policy_annual_premium' in data.columns:
    data['claim_to_premium_ratio'] = data['total_claim_amount'] / data['policy_annual_premium']
    numerical_cols.append('claim_to_premium_ratio')

if 'policy_effective_date' in data.columns and 'incident_date' in data.columns:
    if data['policy_effective_date'].dtype == 'object':
        data['policy_effective_date'] = pd.to_datetime(data['policy_effective_date'])
    if data['incident_date'].dtype == 'object':
        data['incident_date'] = pd.to_datetime(data['incident_date'])
    
    data['customer_tenure_days'] = (data['incident_date'] - data['policy_effective_date']).dt.days
    numerical_cols.append('customer_tenure_days')

if 'incident_state' in data.columns:
    state_fraud_rates = data.groupby('incident_state')[target_col].mean()
    high_risk_states = state_fraud_rates[state_fraud_rates > state_fraud_rates.median()].index
    data['high_risk_geography'] = data['incident_state'].isin(high_risk_states).astype(int)
    numerical_cols.append('high_risk_geography')

print("Feature engineering completed.")

X = data.drop(target_col, axis=1)
y = data[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Fraud rate: {y.mean():.4f}")


Test phase___
Using 'BasePolicy' as target variable
Categorical columns: 23
Numerical columns: 9
Categorical variables encoded.
Feature engineering completed.
Features shape: (15420, 32)
Target shape: (15420,)
Fraud rate: nan
